# 🎨 Vanilla GAN — MNIST Digit Generation

**Dataset:** MNIST — 60,000 grayscale digit images (28×28)  
**Task:** Learn to generate realistic handwritten digits from random noise  

## The GAN Framework

Two networks compete in a minimax game:

| Network | Input | Output | Goal |
|---|---|---|---|
| Generator G | Noise z ~ N(0,1) | Fake image | Fool D |
| Discriminator D | Real or fake image | P(real) ∈ (0,1) | Detect fakes |

## Objective Functions

**Discriminator** maximizes:
$$\mathcal{L}_D = \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

**Generator** minimizes:
$$\mathcal{L}_G = \mathbb{E}[\log(1 - D(G(z)))]$$

In practice we use the non-saturating variant for G:
$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z))]$$

## Architecture

**Generator:** $z(100) \to 256 \to 512 \to 1024 \to 784$ + BN + LeakyReLU + Tanh  
**Discriminator:** $784 \to 512 \to 256 \to 1$ + LeakyReLU + Dropout + Sigmoid

## Key Engineering Decisions

- **Label smoothing 0.9**: real labels = 0.9 not 1.0 — prevents D overconfidence  
- **`fake.detach()`**: stops gradients flowing into G during D's backward pass  
- **BatchNorm in G, not D**: stabilizes G without corrupting D's distribution estimates  
- **Adam betas=(0.5, 0.999)**: reduced beta1 prevents oscillation in GAN training

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision
from PIL import Image

from src.vanilla_gan.generator     import Generator
from src.vanilla_gan.discriminator import Discriminator
from src.vanilla_gan.trainer       import VanillaGANTrainer

with open("../configs/vanilla_gan_config.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")
print(f"\nDevice: {device}")

In [ ]:
# Inspect model architectures before training
G = Generator(
    latent_dim  = config["model"]["latent_dim"],
    hidden_dims = config["model"]["hidden_dims"],
    image_dim   = config["model"]["image_dim"],
).to(device)

D = Discriminator(image_dim=config["model"]["image_dim"]).to(device)

print("Generator architecture:")
print(G)
print(f"\nGenerator parameters  : {G.count_parameters():,}")
print(f"Discriminator parameters: {D.count_parameters():,}")

# Verify forward pass shapes
z_test    = torch.randn(4, config["model"]["latent_dim"]).to(device)
fake_test = G(z_test)
d_test    = D(fake_test)

print(f"\nShape check:")
print(f"  z          : {z_test.shape}")
print(f"  G(z)       : {fake_test.shape}")
print(f"  D(G(z))    : {d_test.shape}")

In [ ]:
# Train the model
# CPU: ~5 mins per epoch at batch_size=64
# GPU: ~30 seconds per epoch
# Increase epochs to 50+ on GPU for quality results

trainer = VanillaGANTrainer(config)
trainer.train()

history = trainer.get_history()
print(f"\nTraining complete.")
print(f"Final D Loss : {history['d_losses'][-1]:.4f}")
print(f"Final G Loss : {history['g_losses'][-1]:.4f}")

In [ ]:
# Plot training loss curves
history = trainer.get_history()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["d_losses"], color="#E74C3C", linewidth=1.8, label="D Loss")
ax.plot(history["g_losses"], color="#1A6BC4", linewidth=1.8, label="G Loss")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss",  fontsize=12)
ax.set_title("Vanilla GAN — Training Loss Curves", fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nHealthy training signs:")
print("  D loss stabilizes (not collapsing to 0)")
print("  G loss decreases as generator improves")
print("  Neither loss explodes — no mode collapse")

In [ ]:
# Visualize generated images from the trained Generator
trainer.G.eval()
with torch.no_grad():
    z    = torch.randn(64, config["model"]["latent_dim"]).to(trainer.device)
    fake = trainer.G(z)                                    # (64, 1, 28, 28)

# make_grid: arranges 64 images into 8x8 grid
# normalize=True: maps [-1,1] → [0,1] for display
grid = torchvision.utils.make_grid(fake, nrow=8, normalize=True, value_range=(-1, 1))
grid_np = grid.permute(1, 2, 0).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid_np, cmap="gray")
ax.axis("off")
ax.set_title(f"Generated MNIST Digits (after {config['training']['epochs']} epochs)",
             fontsize=13)
plt.tight_layout()
os.makedirs("../assets/vanilla_gan", exist_ok=True)
plt.savefig("../assets/vanilla_gan/notebook_generated_grid.png", dpi=150)
plt.show()
print("Saved → assets/vanilla_gan/notebook_generated_grid.png")

In [ ]:
# Show generator quality progression across saved epochs
# Loads per-epoch grids saved to samples/vanilla_gan/ during training

samples_dir = "../samples/vanilla_gan"
epoch_files = sorted([
    f for f in os.listdir(samples_dir) if f.endswith(".png")
])

if not epoch_files:
    print("No epoch samples found. Run training first.")
else:
    # show up to 5 evenly spaced epochs
    indices = np.linspace(0, len(epoch_files) - 1, min(5, len(epoch_files)), dtype=int)
    selected = [epoch_files[i] for i in indices]

    fig, axes = plt.subplots(1, len(selected), figsize=(4 * len(selected), 4))
    if len(selected) == 1:
        axes = [axes]

    for ax, fname in zip(axes, selected):
        epoch_num = fname.replace("epoch_", "").replace(".png", "")
        img = Image.open(os.path.join(samples_dir, fname))
        ax.imshow(img, cmap="gray")
        ax.set_title(f"Epoch {epoch_num}", fontsize=11)
        ax.axis("off")

    plt.suptitle("Generator Quality Over Training", fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f"Showing {len(selected)} of {len(epoch_files)} saved epochs")

## Results Summary

| Metric | Value |
|--------|-------|
| Final D Loss | *see output above* |
| Final G Loss | *see output above* |
| Training Epochs | see config |

**Key Takeaways:**
- **GAN equilibrium**: healthy training converges to D loss ≈ 1.386 (= -2·log(0.5))  
- **Label smoothing**: using 0.9 instead of 1.0 keeps gradients flowing to G throughout  
- **`fake.detach()`**: without this, D's backward pass wastefully updates G's weights  
- **Mode collapse warning**: if G loss drops to 0 and D loss spikes, the generator  
  has collapsed to producing one image — reduce learning rate and retrain